In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Output folder ─────────────────────────────────────────────────────
OUTDIR = Path('figures')
OUTDIR.mkdir(exist_ok=True)

# ── Global style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         7.5,
    'axes.titlesize':    8.5,
    'axes.labelsize':    7.5,
    'xtick.labelsize':   6.8,
    'ytick.labelsize':   6.8,
    'legend.fontsize':   6.5,
    'axes.linewidth':    0.7,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'grid.linewidth':    0.4,
    'grid.alpha':        0.45,
    'figure.dpi':        300,
})

# ── Palette ───────────────────────────────────────────────────────────
C = {
    'raw':     '#1B2A3B',
    'vit':     '#2471A3',
    'uni':     '#117A65',
    'per':     '#B7950B',
    'before':  '#1F618D',
    'after':   '#117A65',
    'lat_raw': '#1B2A3B',
    'lat_uni': '#117A65',
    'grid':    '#BDC3C7',
    'annot':   '#2C3E50',
}

# ═══════════════════════════════════════════════════════════════════════
# ── DATA — final corrected results (2026-03-04 report)
# ═══════════════════════════════════════════════════════════════════════

subjects = ['S1','S2','S3','S4','S5','S6','S7','S8','S9','S10']

# Raw argmax per subject (from report)
raw_acc   = [96.93, 94.48, 96.03, 88.79, 96.80, 89.63, 96.15, 96.55, 94.32, 98.39]
raw_fcmin = [ 1.73,  4.46,  4.10,  9.41,  5.59,  5.65,  5.12,  4.10,  7.25,  0.98]
raw_lat   = [ 10.0,  10.0,  10.0,  10.0,  10.0,  10.0,  10.0,  10.0,  10.0,  10.0]

# Viterbi + Uniform sweep per subject
uni_acc   = [94.62, 92.22, 92.24, 87.02, 92.79, 88.81, 94.19, 94.56, 92.07, 94.19]
uni_fcmin = [ 1.43,  3.60,  4.10,  8.53,  2.75,  5.25,  4.52,  2.46,  5.21,  0.55]
uni_lat   = [ 40.0,  40.0,  60.0,  40.0,  60.0,  20.0,  40.0,  40.0,  40.0,  60.0]

# Aggregate means over 10 LOSO folds
methods   = ['Raw\nArgmax', 'Viterbi\nOnly', 'Vit + Unif\nSweep', 'Vit + Per-State\nOpt']
agg_acc   = [94.81, 94.82, 92.27, 91.69]
agg_f1    = [0.9457, 0.9459, 0.9209, 0.9147]
agg_fcmin = [4.84,   4.84,   3.84,   3.67]
agg_lat   = [10.0,   10.0,   44.0,   46.0]

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 2 — Grouped bar chart: 4 metrics × 4 methods
# ═══════════════════════════════════════════════════════════════════════

fig2, axes = plt.subplots(2, 2, figsize=(7.16, 4.8))
fig2.patch.set_facecolor('white')
axes = axes.flatten()   # iterate as flat list

colors = [C['raw'], C['vit'], C['uni'], C['per']]
x      = np.arange(len(methods))
bw     = 0.58

metric_data   = [agg_acc, [v*100 for v in agg_f1], agg_fcmin, agg_lat]
metric_labels = ['Accuracy (%)', 'Macro-F1 (×100)', 'FC/min', 'Latency (ms)']
metric_ylims  = [(89, 96.5), (89, 96), (3.0, 5.5), (5, 52)]
higher_better = [True, True, False, False]

for ax, data, ylabel, ylim, hb in zip(
        axes, metric_data, metric_labels, metric_ylims, higher_better):

    bars = ax.bar(x, data, width=bw, color=colors,
                  edgecolor='white', linewidth=0.6, zorder=3)

    for bar, val in zip(bars, data):
        if ylabel == 'FC/min':
            fmt = f'{val:.2f}'
        elif ylabel == 'Macro-F1 (×100)':
            fmt = f'{val:.1f}'
        else:
            fmt = f'{val:.1f}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (ylim[1]-ylim[0])*0.012,
                fmt, ha='center', va='bottom',
                fontsize=6.0, color=C['annot'])

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=6.2, rotation=25,
                       ha='right', rotation_mode='anchor')
    ax.set_ylabel(ylabel, labelpad=3)
    ax.set_ylim(ylim)
    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', length=0)

legend_patches = [mpatches.Patch(color=c, label=l) for c, l in
                  zip(colors, ['Raw Argmax', 'Viterbi Only',
                                'Vit + Uniform Sweep', 'Vit + Per-State Opt'])]
fig2.legend(handles=legend_patches, loc='upper center',
            ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03),
            fontsize=6.5, handlelength=1.2,
            handletextpad=0.5, columnspacing=1.0)

plt.tight_layout(rect=[0, 0.01, 1, 0.95])
fig2.savefig(OUTDIR / 'fig2_postfilter_comparison.pdf', dpi=300,
             bbox_inches='tight', facecolor='white')
fig2.savefig(OUTDIR / 'fig2_postfilter_comparison.png', dpi=300,
             bbox_inches='tight', facecolor='white')
print("Fig 2 saved")

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 3 — Per-subject FC/min: Raw vs Uniform Sweep
# ═══════════════════════════════════════════════════════════════════════

fig3, ax3 = plt.subplots(figsize=(7.16, 2.55))
fig3.patch.set_facecolor('white')

x3  = np.arange(len(subjects))
bw3 = 0.35

ax3.bar(x3 - bw3/2, raw_fcmin, bw3, color=C['before'],
        label='Raw Argmax', edgecolor='white', linewidth=0.6, zorder=3)
ax3.bar(x3 + bw3/2, uni_fcmin, bw3, color=C['after'],
        label='Vit + Uniform Sweep', edgecolor='white', linewidth=0.6, zorder=3)

ax3.set_xticks(x3)
ax3.set_xticklabels(subjects)
ax3.set_ylabel('False Changes per Minute (FC/min)')
ax3.set_ylim(0, 11.0)
ax3.yaxis.grid(True, zorder=0)
ax3.set_axisbelow(True)
ax3.legend(loc='upper right', frameon=False, fontsize=6.5)

mean_raw = np.mean(raw_fcmin)
mean_uni = np.mean(uni_fcmin)
ax3.axhline(mean_raw, color=C['before'], lw=0.8, linestyle='--', alpha=0.6)
ax3.axhline(mean_uni, color=C['after'],  lw=0.8, linestyle='--', alpha=0.6)
ax3.text(9.55, mean_raw + 0.15, f'μ={mean_raw:.2f}',
         fontsize=5.2, color=C['before'], ha='right')
ax3.text(9.55, mean_uni - 0.40, f'μ={mean_uni:.2f}',
         fontsize=5.2, color=C['after'], ha='right')

plt.tight_layout(rect=[0, 0.07, 1, 1.0])
fig3.savefig(OUTDIR / 'fig3_persubject_fcmin.pdf', dpi=300,
             bbox_inches='tight', facecolor='white')
fig3.savefig(OUTDIR / 'fig3_persubject_fcmin.png', dpi=300,
             bbox_inches='tight', facecolor='white')
print("Fig 3 saved")

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 4 — Per-subject transition latency: Raw vs Uniform Sweep
# (conceptual inset removed — latency now increases as physically expected)
# ═══════════════════════════════════════════════════════════════════════

fig4, ax4 = plt.subplots(figsize=(7.16, 2.6))
fig4.patch.set_facecolor('white')

x4  = np.arange(len(subjects))
bw4 = 0.35

ax4.bar(x4 - bw4/2, raw_lat, bw4, color=C['lat_raw'],
        label='Raw Argmax', edgecolor='white', linewidth=0.6, zorder=3)
ax4.bar(x4 + bw4/2, uni_lat, bw4, color=C['lat_uni'],
        label='Vit + Uniform Sweep', edgecolor='white', linewidth=0.6, zorder=3)

mean_raw_lat = np.mean(raw_lat)
mean_uni_lat = np.mean(uni_lat)
ax4.axhline(mean_raw_lat, color=C['lat_raw'], lw=0.8, linestyle='--', alpha=0.6)
ax4.axhline(mean_uni_lat, color=C['lat_uni'], lw=0.8, linestyle='--', alpha=0.6)
ax4.text(10, mean_raw_lat + 0.81, f'μ={mean_raw_lat:.0f} ms',
         fontsize=5.2, color=C['lat_raw'], ha='right')
ax4.text(10, mean_uni_lat + 0.81, f'μ={mean_uni_lat:.0f} ms',
         fontsize=5.2, color=C['lat_uni'], ha='right')

ax4.set_xticks(x4)
ax4.set_xticklabels(subjects)
ax4.set_ylabel('Transition Detection Latency (ms)')
ax4.set_ylim(0, 75)
ax4.yaxis.grid(True, zorder=0)
ax4.set_axisbelow(True)
ax4.legend(loc='upper right', frameon=False, fontsize=6.5)

plt.tight_layout(rect=[0, 0.07, 1, 1.0])
fig4.savefig(OUTDIR / 'fig4_latency1.pdf', dpi=300,
             bbox_inches='tight', facecolor='white')
fig4.savefig(OUTDIR / 'fig4_latency1.png', dpi=300,
             bbox_inches='tight', facecolor='white')
print("Fig 4 saved")
print("All figures done.")

Fig 2 saved
Fig 3 saved
Fig 4 saved
All figures done.
